<img src="https://s3.amazonaws.com/weclouddata/images/logos/wcd_logo_new_2.png" width="10%">

<h1><center>BERT Use Cases (Fine-tuning)</center></h1>

## Introduction

**Fine-tuning BERT:**

BERT, pre-trained on vast textual data, captures rich linguistic structures and nuances, making it a powerful base for various NLP tasks.

1. **Utilizing the Pre-trained BERT**: This model, having been trained on massive corpora, understands intricate language details, saving significant time and computational effort compared to training from scratch.
2. **Adding Task-specific Layers**: For specific tasks, like classification, additional layers (e.g., a softmax layer) are appended.
3. **Training on Task-specific Data**: While retaining the knowledge from its pre-training, BERT's weights are further optimized using task-specific data.

**Benefits**:

- **Efficiency**: Leveraging a pre-trained model drastically reduces training time.
- **Superior Performance**: Using models pre-trained on large corpora often yields better results, especially when task-specific data is scarce.
- **Versatility**: One BERT model can be adapted for multiple tasks through fine-tuning.

## Load Model

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
import numpy as np
from pprint import pprint
from transformers import BertTokenizerFast, BertModel

In [ ]:
# Check if GPU is available
print("GPU Available:", torch.cuda.is_available())
# Which GPU are we using
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
# Select device (GPU if available, otherwise CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


GPU Available: True
GPU Name: Tesla T4
Using device: cuda


In [ ]:
tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [ ]:
bert = BertModel.from_pretrained("bert-base-uncased")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
bert = bert.to(device)

## Document-level

We can use the following custom classification model to perform document classification tasks leveraging on pre-trained BERT embedding.

In [ ]:
class BERTForClassification(nn.Module):
    def __init__(self, bert, n_class):
        super().__init__()

        # Ensure that the number of classes is more than one.
        assert n_class > 1, ValueError("number of class must larger than 1")

        # Number of classes for the classification task.
        self.n_class = n_class

        # The pre-trained BERT model.
        self.bert = bert

        # Define the output layer. It's a simple linear layer that maps from the hidden size
        # of the BERT model to the number of classes.
        self.output_layer = nn.Linear(self.bert.config.hidden_size, n_class)

    def forward(self, inputs):

        # Pass the inputs through the BERT model.
        outputs = self.bert(**inputs)

        # Extract the embeddings for the [CLS] token, since we want to do the document-level classification
        # The first element in the sequence is always the [CLS] in BERT
        #                                   [batch, seq_length, emb ]
        cls_emb = outputs.last_hidden_state[:,     0,          :]

        # Pass the [CLS] embeddings through the output layer to produce class logits.
        return self.output_layer(cls_emb) # -> [batch, n_class]

### Document Classification

Let's play with a simple dataset

In [ ]:
X = ["this is a positive example.", "this is a negative example."]
Y = [1, 0]

# Instantiate the custom BERT classifier for binary classification
doc_cls_model = BERTForClassification(bert, 2).to(device)

# Set up optimizer and loss for training
# The learning rate is set to 5e-5, which is commonly used with BERT.
optim = torch.optim.AdamW(doc_cls_model.parameters(), lr=5e-5)
critera = nn.CrossEntropyLoss()

# Set model to training mode
doc_cls_model.train()

# Tokenize input data for BERT
inputs = tokenizer(X, return_tensors="pt", padding=True).to(device)

# Training loop for 20 iterations
for iter in range(20):
    optim.zero_grad()

    # Get model predictions
    logits = doc_cls_model(inputs)
    # Calculate loss
    loss = critera(logits, torch.tensor(Y).to(device))

    print(loss)
    loss.backward()
    optim.step()

tensor(0.5492, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5244, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7028, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6155, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3855, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.2823, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.2458, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.2323, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0523, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0501, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0326, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0285, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0184, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0063, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0079, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0035, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0048, device='cuda:0', grad_fn=

In [ ]:
X = ['I feel good', 'I am so sad']

doc_cls_model.eval()
with torch.no_grad():
    inputs = tokenizer(X, return_tensors='pt', padding=True).to(device)
    logits = doc_cls_model(inputs)

logits

tensor([[-1.2606,  1.5577],
        [-0.1184, -0.1300]], device='cuda:0')

In [ ]:
logits.argmax(dim=1)

tensor([1, 0], device='cuda:0')

### Document Pair Classification

We can use the same classification model to perform the document pair classification.

Let's try on a simple example

In [ ]:
X = [("weekend",'Saturday and Sunday'),('good','random words')]
Y = [1, 0]

docp_cls_model = BERTForClassification(bert, 2).to(device)

optim = torch.optim.AdamW(docp_cls_model.parameters(), lr=5e-5)
critera = nn.CrossEntropyLoss()

docp_cls_model.train()
inputs = tokenizer(X, return_tensors="pt", padding=True).to(device)

for iter in range(20):
    optim.zero_grad()
    logits = docp_cls_model(inputs)

    loss = critera(logits, torch.tensor(Y).to(device))
    print(loss)
    loss.backward()
    optim.step()

tensor(1.0808, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.5131, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.1711, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0571, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0327, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0216, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0150, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0125, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0083, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0032, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0051, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0036, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0030, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0064, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0025, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0021, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0017, device='cuda:0', grad_fn=

In [ ]:
X = [('weekday','monday')]

docp_cls_model.eval()
with torch.no_grad():
    inputs = tokenizer(X, return_tensors='pt', padding=True).to(device)
    logits = docp_cls_model(inputs)

logits

tensor([[-2.8421,  2.9032]], device='cuda:0')

In [ ]:
logits.argmax(dim=1)

tensor([1], device='cuda:0')

**We can also work with the benchmark dataset**

In [ ]:
import os
if not os.path.exists('QNLIv2.zip'):
    !wget https://dl.fbaipublicfiles.com/glue/data/QNLIv2.zip
    !unzip QNLIv2.zip

--2025-10-01 11:00:49--  https://dl.fbaipublicfiles.com/glue/data/QNLIv2.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 13.35.202.17, 13.35.202.124, 13.35.202.30, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|13.35.202.17|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10627589 (10M) [application/zip]
Saving to: ‘QNLIv2.zip’

QNLIv2.zip          100%[===================>]  10.13M  7.06MB/s    in 1.4s    

2025-10-01 11:00:51 (7.06 MB/s) - ‘QNLIv2.zip’ saved [10627589/10627589]

Archive:  QNLIv2.zip
   creating: QNLI/
  inflating: QNLI/dev.tsv            
  inflating: QNLI/test.tsv           
  inflating: QNLI/train.tsv          


In [ ]:
X = []
Y = []
with open('QNLI/dev.tsv', encoding='utf-8') as f:
    f.readline()
    for line in f:
        idx, question, sentence, label = line.rstrip().split('\t')
        X.append((question, sentence))
        Y.append(1 if label == 'entailment' else 0)
len(X)

5463

In [ ]:
X_train = X[:-500]
Y_train = Y[:-500]

In [ ]:
batch_size = 32

my_model = BERTForClassification(bert, 2).to(device)

optim = torch.optim.AdamW(my_model.parameters(), lr=5e-5)
critera = nn.CrossEntropyLoss()

my_model.train()

# Here we will perform batch training
# Training loop for 10 epochs
for i in range(10):
    # Iterating over the training dataset in mini-batches
    for batch_idx in range(0, 1000, batch_size):

        # Extracting data for the current batch
        batch_X = X_train[batch_idx:batch_idx+batch_size]
        batch_Y = Y_train[batch_idx:batch_idx+batch_size]

        optim.zero_grad()
        inputs = tokenizer(batch_X, return_tensors='pt', padding=True, truncation=True).to(device)
        logits = my_model(inputs)

        loss = critera(logits, torch.tensor(batch_Y).to(device))
        loss.backward()
        print(batch_idx, loss)
        optim.step()

0 tensor(0.6819, device='cuda:0', grad_fn=<NllLossBackward0>)
32 tensor(0.6459, device='cuda:0', grad_fn=<NllLossBackward0>)
64 tensor(0.7005, device='cuda:0', grad_fn=<NllLossBackward0>)
96 tensor(0.7056, device='cuda:0', grad_fn=<NllLossBackward0>)
128 tensor(0.6362, device='cuda:0', grad_fn=<NllLossBackward0>)
160 tensor(0.7007, device='cuda:0', grad_fn=<NllLossBackward0>)
192 tensor(0.6231, device='cuda:0', grad_fn=<NllLossBackward0>)
224 tensor(0.7161, device='cuda:0', grad_fn=<NllLossBackward0>)
256 tensor(0.7298, device='cuda:0', grad_fn=<NllLossBackward0>)
288 tensor(0.6570, device='cuda:0', grad_fn=<NllLossBackward0>)
320 tensor(0.6269, device='cuda:0', grad_fn=<NllLossBackward0>)
352 tensor(0.4016, device='cuda:0', grad_fn=<NllLossBackward0>)
384 tensor(0.7263, device='cuda:0', grad_fn=<NllLossBackward0>)
416 tensor(0.8930, device='cuda:0', grad_fn=<NllLossBackward0>)
448 tensor(0.5706, device='cuda:0', grad_fn=<NllLossBackward0>)
480 tensor(0.4981, device='cuda:0', grad_fn=<

In [ ]:
X_test = X[-500:]
Y_test = Y[-500:]

my_model = my_model.eval()

Y_pred = []
batch_size = 16
with torch.no_grad():
    for batch_idx in range(0, 500, batch_size):
        batch_X = X_test[batch_idx:batch_idx+batch_size]

        optim.zero_grad()
        inputs = tokenizer(batch_X, return_tensors='pt', padding=True, truncation=True).to(device)
        logits = my_model(inputs).detach().cpu()

        pred = logits.argmax(dim=1).numpy().tolist()
        print(batch_idx, pred)
        Y_pred.extend(pred)

0 [1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1]
16 [1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1]
32 [0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1]
48 [1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1]
64 [0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0]
80 [0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1]
96 [0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1]
112 [1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 0]
128 [1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0]
144 [0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0]
160 [1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1]
176 [1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1]
192 [0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0]
208 [0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1]
224 [1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1]
240 [1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1]
256 [1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0]
272 [0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1]
288 [0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0]
3

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(Y_test, Y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.8883    0.6958    0.7804       263
           1     0.7279    0.9030    0.8060       237

    accuracy                         0.7940       500
   macro avg     0.8081    0.7994    0.7932       500
weighted avg     0.8123    0.7940    0.7925       500



## Token-level

To perform the token-level， we just need to change the previous classidier a little bit


In [ ]:
class BERTForTokenClassification(nn.Module):
    def __init__(self, bert, n_class):
        super().__init__()

        # Ensure that the number of classes is more than one.
        assert n_class > 1, ValueError("number of class must larger than 1")

        # Number of classes for the classification task.
        self.n_class = n_class

        # The pre-trained BERT model.
        self.bert = bert

        # Define the output layer. It's a simple linear layer that maps from the hidden size
        # of the BERT model to the number of classes.
        self.output_layer = nn.Linear(self.bert.config.hidden_size, n_class)

    def forward(self, inputs):

        # Pass the inputs through the BERT model.
        outputs = self.bert(**inputs)

        # *************************************************************
        # * Here we take all the embeddings, not just the [cls]       *
        # * This is the only different compare to previous classifier *
        # *************************************************************
        token_embs = outputs.last_hidden_state

        # Pass the embeddings through the output layer to produce class logits.
        return self.output_layer(token_embs) # -> [batch, n_class]

### Token Extraction

#### Prepare the Dataset

In [ ]:
X = [
    "WeCloudData provides high quiality Data Science and AI courses in Toronto.",
]


inputs = tokenizer(X, return_tensors="pt")
[(i, tokenizer.decode(x)) for i, x in enumerate(inputs["input_ids"][0])]

[(0, '[CLS]'),
 (1, 'we'),
 (2, '##cl'),
 (3, '##oud'),
 (4, '##da'),
 (5, '##ta'),
 (6, 'provides'),
 (7, 'high'),
 (8, 'qui'),
 (9, '##ality'),
 (10, 'data'),
 (11, 'science'),
 (12, 'and'),
 (13, 'ai'),
 (14, 'courses'),
 (15, 'in'),
 (16, 'toronto'),
 (17, '.'),
 (18, '[SEP]')]

#### Define my entity types
- WeCloudData -> org -> 1
- Data Science -> topic -> 2
- AI -> topic -> 2
- Toronto -> location -> 3

We define 4 class for each token for 3 entity types and one for empty entity.

In [ ]:
# initial labels
labels = torch.zeros(size=(1,len(inputs['input_ids'][0])), dtype=torch.long)

labels

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [ ]:
labels[0, 1:6] = 1    # -> weclouddata
labels[0, 10:12] = 2  # -> data science
labels[0, 13:14] = 2  # -> ai
labels[0, 16:17] = 3  # -> toronto
labels

tensor([[0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 2, 2, 0, 2, 0, 0, 3, 0, 0]])

In [ ]:
for i in range(inputs['input_ids'][0].shape[0]):
    print(i, labels[0][i], tokenizer.decode(inputs['input_ids'][0][i]))

0 tensor(0) [CLS]
1 tensor(1) we
2 tensor(1) ##cl
3 tensor(1) ##oud
4 tensor(1) ##da
5 tensor(1) ##ta
6 tensor(0) provides
7 tensor(0) high
8 tensor(0) qui
9 tensor(0) ##ality
10 tensor(2) data
11 tensor(2) science
12 tensor(0) and
13 tensor(2) ai
14 tensor(0) courses
15 tensor(0) in
16 tensor(3) toronto
17 tensor(0) .
18 tensor(0) [SEP]


#### train model

In [ ]:
token_model = BERTForTokenClassification(bert, 4).to(device)

optim = torch.optim.AdamW(token_model.parameters(), lr=5e-5)
critera = nn.CrossEntropyLoss()

token_model.train()
inputs = tokenizer(X, return_tensors="pt", padding=True).to(device)

for iter in range(50):
    optim.zero_grad()
    logits = token_model(inputs)
    loss = critera(logits.view(-1, token_model.n_class), labels.view(-1).to(device))
    print(loss)
    loss.backward()
    optim.step()

tensor(1.4307, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.8440, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.6344, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4467, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3429, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.2305, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.1571, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.1272, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0827, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0592, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0649, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0330, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0260, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0212, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0188, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0126, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0092, device='cuda:0', grad_fn=

In [ ]:
test_X = ["2451232 alberta CORP. is an private company based on calgary. Data Science AI powered solutions to its clients."]

token_model.eval()

with torch.no_grad():
    inputs = tokenizer(test_X, return_tensors='pt', padding=True).to(device)
    logits = token_model(inputs)

In [ ]:
for logit, token_id in zip(logits[0].argmax(dim=1), inputs['input_ids'][0]):
    print(logit, tokenizer.decode(token_id))

tensor(0, device='cuda:0') [CLS]
tensor(1, device='cuda:0') 245
tensor(1, device='cuda:0') ##12
tensor(1, device='cuda:0') ##32
tensor(1, device='cuda:0') alberta
tensor(1, device='cuda:0') corp
tensor(0, device='cuda:0') .
tensor(0, device='cuda:0') is
tensor(0, device='cuda:0') an
tensor(0, device='cuda:0') private
tensor(0, device='cuda:0') company
tensor(0, device='cuda:0') based
tensor(0, device='cuda:0') on
tensor(3, device='cuda:0') calgary
tensor(0, device='cuda:0') .
tensor(2, device='cuda:0') data
tensor(2, device='cuda:0') science
tensor(2, device='cuda:0') ai
tensor(0, device='cuda:0') powered
tensor(0, device='cuda:0') solutions
tensor(0, device='cuda:0') to
tensor(0, device='cuda:0') its
tensor(0, device='cuda:0') clients
tensor(0, device='cuda:0') .
tensor(0, device='cuda:0') [SEP]


### Token Classification - BIO

#### BIO tagging

BIO tagging is a common tagging format for tagging tokens in a chunking task.

- B: Begining
- I: Inside
- O: Outside

e.g.

WeCloudData provides high quiality Data Science and AI courses in Toronto.

- O -> [CLS]
- B-org -> we
- I-org -> ##cl
- I-org -> ##oud
- I-org -> ##da
- I-org -> ##ta
- O -> provides
- O -> high
- O -> qui
- O -> ##ality
- B-topic -> data
- I-topic -> science
- O -> and
- B-topic -> ai
- O -> courses
- O -> in
- B-loc -> toronto
- O -> .
- O -> [SEP]

In [ ]:
X = [
    "WeCloudData provides high quiality Data Science and AI courses in Toronto.",
]


inputs = tokenizer(X, return_tensors="pt")
[(i, tokenizer.decode(x)) for i, x in enumerate(inputs["input_ids"][0])]

[(0, '[CLS]'),
 (1, 'we'),
 (2, '##cl'),
 (3, '##oud'),
 (4, '##da'),
 (5, '##ta'),
 (6, 'provides'),
 (7, 'high'),
 (8, 'qui'),
 (9, '##ality'),
 (10, 'data'),
 (11, 'science'),
 (12, 'and'),
 (13, 'ai'),
 (14, 'courses'),
 (15, 'in'),
 (16, 'toronto'),
 (17, '.'),
 (18, '[SEP]')]

In [ ]:
labels = torch.zeros(size=(1, len(inputs['input_ids'][0])), dtype=torch.long)

labels

tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

In [ ]:
labels[0, 1] = 1
labels[0,2: 6] = 2
labels[0, 10] = 1
labels[0, 11:12] = 2
labels[0, 13] = 1
labels[0, 16] = 1
labels

tensor([[0, 1, 2, 2, 2, 2, 0, 0, 0, 0, 1, 2, 0, 1, 0, 0, 1, 0, 0]])

In [ ]:
for i in range(inputs['input_ids'][0].shape[0]):
    print(i, labels[0][i], tokenizer.decode(inputs['input_ids'][0][i]))


0 tensor(0) [CLS]
1 tensor(1) we
2 tensor(2) ##cl
3 tensor(2) ##oud
4 tensor(2) ##da
5 tensor(2) ##ta
6 tensor(0) provides
7 tensor(0) high
8 tensor(0) qui
9 tensor(0) ##ality
10 tensor(1) data
11 tensor(2) science
12 tensor(0) and
13 tensor(1) ai
14 tensor(0) courses
15 tensor(0) in
16 tensor(1) toronto
17 tensor(0) .
18 tensor(0) [SEP]


In [ ]:
model = BERTForTokenClassification(bert, 4).to(device)

optim = torch.optim.AdamW(model.parameters(), lr=5e-5)
critera = nn.CrossEntropyLoss()

model.train()
inputs = tokenizer(X, return_tensors="pt", padding=True).to(device)
for iter in range(10):
    optim.zero_grad()
    logits = model(inputs)

    loss = critera(logits.view(-1, model.n_class), labels.view(-1).to(device))
    print(loss)
    loss.backward()
    optim.step()

tensor(1.4007, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.7571, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.4465, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.3248, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.2475, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.1725, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.1478, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0903, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0792, device='cuda:0', grad_fn=<NllLossBackward0>)
tensor(0.0532, device='cuda:0', grad_fn=<NllLossBackward0>)


In [ ]:
X = ["ServiceNow is an public company based on the states. It has offices in Toronto and Calgary, and BC. It provides AI powered solutions to its clients."]

model.eval()
with torch.no_grad():
    inputs = tokenizer(X, return_tensors='pt', padding=True).to(device)
    logits = model(inputs)

In [ ]:
for logit, token_id in zip(logits[0].argmax(dim=1), inputs['input_ids'][0]):
    print(logit, tokenizer.decode(token_id))

tensor(0, device='cuda:0') [CLS]
tensor(1, device='cuda:0') service
tensor(2, device='cuda:0') ##now
tensor(0, device='cuda:0') is
tensor(0, device='cuda:0') an
tensor(0, device='cuda:0') public
tensor(0, device='cuda:0') company
tensor(0, device='cuda:0') based
tensor(0, device='cuda:0') on
tensor(0, device='cuda:0') the
tensor(0, device='cuda:0') states
tensor(0, device='cuda:0') .
tensor(0, device='cuda:0') it
tensor(0, device='cuda:0') has
tensor(0, device='cuda:0') offices
tensor(0, device='cuda:0') in
tensor(1, device='cuda:0') toronto
tensor(0, device='cuda:0') and
tensor(1, device='cuda:0') calgary
tensor(0, device='cuda:0') ,
tensor(0, device='cuda:0') and
tensor(1, device='cuda:0') bc
tensor(0, device='cuda:0') .
tensor(0, device='cuda:0') it
tensor(0, device='cuda:0') provides
tensor(1, device='cuda:0') ai
tensor(0, device='cuda:0') powered
tensor(0, device='cuda:0') solutions
tensor(0, device='cuda:0') to
tensor(0, device='cuda:0') its
tensor(0, device='cuda:0') clients
ten